# Gradient Boost Regression

In [ ]:
# importing libs:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.calibration import LabelEncoder
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from category_encoders.target_encoder import TargetEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
df = pd.read_csv("../../../../../data/Used_Car_Price_Prediction.csv")
df.head()

## Basic EDA

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
non_numeric_cols = df.select_dtypes(exclude='number').columns
numeric_cols

In [ ]:
non_numeric_cols

In [ ]:
for col in non_numeric_cols:
   print(df[col].value_counts(),"\n")

In [ ]:
cols = df.columns
for col in cols:
   print(col ,':',df[col].count(),":" ,len(df[col].unique()),"\n")
   print(df[col].value_counts(),"\n")

In [ ]:
df.drop(columns=['original_price'], inplace = True) ## to many NaN

In [ ]:
df.describe().T

In [ ]:
num_col = df.select_dtypes(include = 'number')
plt.figure(figsize=(12,40))
index = 1
for col in num_col:
    plt.subplot(11,4,index)
    sns.kdeplot(df[col])
    index +=1

In [ ]:
plt.figure(figsize=(8,6))
sns.lineplot(df['sale_price'])
plt.title('Lineplot for Sale Price')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(num_col.corr(),annot=True,cmap='Blues')

In [ ]:
sns.boxplot(df)

In [ ]:
def cap_outliers(df):
    num = df.select_dtypes(include = 'number')
    for col in num:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower, upper)
    return df 

df = cap_outliers(df)

In [ ]:
sns.pairplot(num_col)

In [ ]:
df.drop(columns='ad_created_on')

In [ ]:
df.info()

In [ ]:
df.columns

## Data Prep

In [ ]:
df['fitness_certificate'] = df['fitness_certificate'].astype('boolean')
df_bool = df.select_dtypes('boolean')
df_bool

In [ ]:
df['car_availability'].unique()
df.drop(columns=['original_price'], inplace = True) ## to many NaN

In [ ]:
df_obj = df.select_dtypes(include='object')
df_obj_one = []
for col in df_obj.columns:
    unique_vals = df[col].dropna().unique()
    n_unique = len(unique_vals)

    if n_unique < 10:
        print(f"\nColumn: {col}")
        df_obj_one.append(col)
        print(f"Number of unique values: {n_unique}")
        print("Unique values:")
        print(unique_vals)
print(f'df_obj_one: {df_obj_one}')

In [ ]:
x = 5

df_obj = df.select_dtypes(include='object')

for col in df_obj.columns:
    n_unique = df[col].nunique(dropna=True)
    if n_unique < x:
        print(f"\nColumn: {col}")
        print(f"Number of unique values: {n_unique}")
        print(df[col].value_counts())


In [ ]:
df_obj_one # to be one hot encoded

In [ ]:
df_obj_target = [x for x in df_obj.columns if x not in df_obj_one]
df_obj_target

## Pipelines And Spit

In [ ]:
X = df.drop(columns = 'sale_price')
y = df['sale_price']

In [ ]:
num_features = X.select_dtypes(include='number')
num_features = num_features.columns
num_features # numeric

In [ ]:
df_obj_target # categorical with >10 unique values

In [ ]:
df_obj_one # categorical with <10 unique values

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline_one = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one', OneHotEncoder(handle_unknown='ignore',drop='first'))
])
cat_pipeline_target = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target', TargetEncoder(handle_unknown='ignore',smoothing=5)),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, num_features),
    ('cat_one',cat_pipeline_one,df_obj_one),
    ('cat_target',cat_pipeline_target,df_obj_target)
])

In [ ]:
models = {
    "Gradient_Boost": GradientBoostingRegressor(),
}

In [ ]:
preprocessor

In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

In [ ]:
pipelines['Gradient_Boost']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42
)

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
# X_train_transformed=pd.DataFrame(X_train)
mew = pd.DataFrame(X_train_transformed)
mew.head()

In [ ]:
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
# Check if there are any NaNs in the transformed data
print(np.isnan(X_train_transformed).sum())  # for training set
print(np.isnan(X_test_transformed).sum())   # for test set


In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
print(pd.DataFrame(X_train_transformed).isna().sum().sum())  # should be 0

X_test_transformed = preprocessor.transform(X_test)
print(pd.DataFrame(X_test_transformed).isna().sum().sum())  # should be 0

# Columns in X_train
train_cols = set(X_train.columns)

# Columns in X_test
test_cols = set(X_test.columns)

# Columns missing in test
missing_in_test = train_cols - test_cols
print("Columns in train but missing in test:", missing_in_test)

# Numeric
print("Missing numeric columns:", [c for c in num_features if c not in X_test.columns])

# One-hot categorical
print("Missing one-hot columns:", [c for c in df_obj_one if c not in X_test.columns])

# Target categorical
print("Missing target-encoded columns:", [c for c in df_obj_target if c not in X_test.columns])

X_test_transformed = preprocessor.transform(X_test)
na_count = pd.DataFrame(X_test_transformed).isna().sum()
print("NaNs per column in transformed test set:")
print(na_count[na_count > 0])

X_test_nan_counts = X_test.isna().sum()
print(X_test_nan_counts[X_test_nan_counts > 0])
import category_encoders as ce

for col in df_obj_target:
    missing_after_te = X_test[col].isna().sum()
    print(f"{col}: {missing_after_te} NaNs in test set before encoding")


## Pipeline redefine:

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline_one = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one', OneHotEncoder(handle_unknown='ignore',drop='first'))
])
cat_pipeline_target = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('target', TargetEncoder(handle_unknown='ignore',smoothing=5)),
    ('final_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, num_features),
    ('cat_one',cat_pipeline_one,df_obj_one),
    ('cat_target',cat_pipeline_target,df_obj_target)
])

In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}
pipelines['Gradient_Boost']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42
)
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
# X_train_transformed=pd.DataFrame(X_train)
mew = pd.DataFrame(X_train_transformed)
mew.head()
X_test_transformed = preprocessor.transform(X_test)
# Check if there are any NaNs in the transformed data
print(np.isnan(X_train_transformed).sum())  # for training set
print(np.isnan(X_test_transformed).sum())   # for test set


# Evaluations and Results

In [ ]:
def get_regression_metrics(y_true, y_pred, model_name=None, verbose=True, plot=True):
    """
    Calculate standard regression metrics and optionally print and visualize them.

    Parameters
    ----------
    y_true : array-like
        True target values
    y_pred : array-like
        Predicted target values
    model_name : str, optional
        Name of the model (for printing)
    verbose : bool
        Whether to print metrics
    plot : bool
        Whether to plot visualizations

    Returns
    -------
    metrics : dict
        Dictionary containing MAE, MSE, RMSE, R2
    """

    # Metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    }

    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"MAE  : {mae:.2f}")
        print(f"MSE  : {mse:.2f}")
        print(f"RMSE : {rmse:.2f}")
        print(f"R2   : {r2:.4f}")
        print("-"*40)

    # Visualizations
    if plot:
        plt.figure(figsize=(16,5))

        # 1️⃣ True vs Predicted Scatter
        plt.subplot(1,2,1)
        sns.scatterplot(x=y_true, y=y_pred, alpha=0.6)
        plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', linewidth=2)
        plt.xlabel("Actual Values")
        plt.ylabel("Predicted Values")
        plt.title(f"Actual vs Predicted {'(' + model_name + ')' if model_name else ''}")

        # 2️⃣ Residual Plot
        plt.subplot(1,2,2)
        residuals = y_true - y_pred
        sns.histplot(residuals, kde=True, bins=30, color='orange')
        plt.xlabel("Residuals")
        plt.title(f"Residuals Distribution {'(' + model_name + ')' if model_name else ''}")

        plt.tight_layout()
        plt.show()

    return metrics


In [ ]:
# Dictionary to store results
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)

    # -------------------------
    # TRAIN metrics
    # -------------------------
    y_train_pred = pipe.predict(X_train)
    train_metrics = get_regression_metrics(
        y_train,
        y_train_pred,
        model_name=f"{name} (Train)",
        verbose=True,
        plot=True
    )

    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=True
    )
    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }


# SOMETHING AMISS

## SOMETHING AMISS

In [ ]:
for col in X_train.select_dtypes(include=np.number).columns:
    if np.allclose(X_train[col].values, y_train.values):
        print("LEAKAGE COLUMN:", col)


In [ ]:
corr = X_train.select_dtypes(include=np.number).corrwith(y_train)
print(corr.sort_values(ascending=False).head(10))

In [ ]:
from sklearn.dummy import DummyRegressor
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
print(r2_score(y_test, dummy.predict(X_test)))


In [ ]:
# identify columns with nearly unique values
n_rows = X_train.shape[0]
unique_counts = X_train.nunique(dropna=False).sort_values(ascending=False)
print(unique_counts.head(30))

# Columns with unique count equal to number of rows
unique_cols = unique_counts[unique_counts >= 0.99 * n_rows].index.tolist()
print("High-cardinality / unique-like cols:", unique_cols)


In [ ]:
from sklearn.linear_model import LinearRegression
suspect_features = []
for col in X_train.select_dtypes(include=np.number).columns:
    xi = X_train[[col]].fillna(0).values
    if np.unique(xi).size < 2:
        continue
    r2 = LinearRegression().fit(xi, y_train).score(xi, y_train)
    if r2 > 0.95:
        suspect_features.append((col, r2))
suspect_features[:20], len(suspect_features)

import pandas as pd
high_corr_cat = []
for col in X_train.select_dtypes(include='object').columns:
    grp = pd.concat([X_train[col], y_train], axis=1).groupby(col)[y_train.name].mean()
    mapped = X_train[col].map(grp).fillna(y_train.mean()).values.reshape(-1,1)
    r2 = LinearRegression().fit(mapped, y_train).score(mapped, y_train)
    if r2 > 0.95:
        high_corr_cat.append((col, r2))
high_corr_cat

from sklearn.metrics import r2_score
lin_relations = []
for col in X_train.columns:
    try:
        xi = X_train[[col]].fillna(0).values.astype(float)
    except Exception:
        continue
    if np.unique(xi).size < 2:
        continue
    a = np.linalg.lstsq(np.hstack([xi, np.ones_like(xi)]), y_train.values, rcond=None)[0]
    y_pred_lin = xi * a[0] + a[1]
    r2 = r2_score(y_train, y_pred_lin)
    if r2 > 0.95:
        lin_relations.append((col, float(r2), float(a[0]), float(a[1])))
lin_relations

print(y_train.describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99]))
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.boxplot(y_train)
plt.title("y_train boxplot")
plt.show()


# TIME SERIES!!!

In [ ]:
df = df.sort_values("ad_created_on")
df['ad_created_on']

In [ ]:
# better alternative:
df["ad_created_on"] = pd.to_datetime(df["ad_created_on"], errors="coerce")
max_date = df['ad_created_on'].max()
df['listing_age_days'] = (max_date - df['ad_created_on']).dt.days
# drop ad_created_on afterwards


In [ ]:
df["ad_created_on"] = pd.to_datetime(df["ad_created_on"], errors="coerce")
df["ad_created_on"]
df["year"] = df["ad_created_on"].dt.year
df["month"] = df["ad_created_on"].dt.month
df["dayofweek"] = df["ad_created_on"].dt.dayofweek
df["hour"] = df["ad_created_on"].dt.hour

In [ ]:
# and cyclic features:
# month in 1..12
df['month_sin'] = np.sin(2*np.pi * (df['month']-1) / 12)
df['month_cos'] = np.cos(2*np.pi * (df['month']-1) / 12)

# hour in 0..23
df['hour_sin'] = np.sin(2*np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2*np.pi * df['hour'] / 24)

# dayofweek in 0..6
df['dow_sin'] = np.sin(2*np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2*np.pi * df['dayofweek'] / 7)


In [ ]:
df = df.drop(columns=["ad_created_on"])

## RESPLIT

In [ ]:
print(df[["year","month"]].corrwith(df["sale_price"]).sort_values())

In [ ]:
X = df.drop(columns = 'sale_price')
y = df['sale_price']
leak_features = ["booking_down_pymnt", "emi_starts_from", "broker_quote"]
X = X.drop(columns=leak_features)

In [ ]:
df_obj = X.select_dtypes(include='object')
df_obj_one = []
for col in df_obj.columns:
    unique_vals = df[col].dropna().unique()
    n_unique = len(unique_vals)

    if n_unique < 10:
        print(f"\nColumn: {col}")
        df_obj_one.append(col)
        print(f"Number of unique values: {n_unique}")
        print("Unique values:")
        print(unique_vals)
print(df_obj_one)
df_obj_target = [x for x in df_obj.columns if x not in df_obj_one]
df_obj_target
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    shuffle=False
)
# now we make the pipelines again:

num_features = X.select_dtypes(include='number')
num_features = num_features.columns
num_features # numeric

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline_one = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one', OneHotEncoder(handle_unknown='ignore',drop='first'))
])
cat_pipeline_target = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('target', TargetEncoder(handle_unknown='ignore',smoothing=5)),
    ('final_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, num_features),
    ('cat_one',cat_pipeline_one,df_obj_one),
    ('cat_target',cat_pipeline_target,df_obj_target)
])
models = {
    "Random Forest": RandomForestRegressor(),
    "Linear Regression": LinearRegression(),
    "Ridge":Ridge(alpha=1.0),
    "Lasso":Lasso(),
    "ElasticNet":ElasticNet(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(),
    "Gradient_Boost": GradientBoostingRegressor()
}
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}

In [ ]:
# Dictionary to store results
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)

    # -------------------------
    # TRAIN metrics
    # -------------------------
    y_train_pred = pipe.predict(X_train)
    train_metrics = get_regression_metrics(
        y_train,
        y_train_pred,
        model_name=f"{name} (Train)",
        verbose=True,
        plot=True
    )

    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=True
    )
    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }


In [ ]:
# Dictionary to store results
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)


    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=True
    )
    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }


In [ ]:
# Dictionary to store results
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)


    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=False
    )
    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }


# REDO!!!

In [ ]:
df = pd.read_csv("../../../../../data/Used_Car_Price_Prediction.csv")
df.head()

In [ ]:
df['source'].value_counts()

In [ ]:
df.drop(columns=['original_price'], inplace = True) ## to many NaN

In [ ]:
df['fitness_certificate'] = df['fitness_certificate'].astype('boolean')
df_bool = df.select_dtypes('boolean')
df_bool

In [ ]:
df = df.sort_values("ad_created_on")

In [ ]:
train = df.iloc[:int(0.8*len(df))].copy()
test = df.iloc[int(0.8*len(df)):].copy()

In [ ]:
# Check for any problematic values in the column
train['ad_created_on'] = pd.to_datetime(train['ad_created_on'], errors='coerce')

# Verify the result
print(train['ad_created_on'].dtypes)  # Should be 'datetime64[ns]'

In [ ]:
# Check for any problematic values in the column
test['ad_created_on'] = pd.to_datetime(test['ad_created_on'], errors='coerce')

# Verify the result
print(test['ad_created_on'].dtypes)  # Should be 'datetime64[ns]'

In [ ]:
# Verify that 'ad_created_on' is indeed a datetime column
print(train['ad_created_on'].dtype)  # Should print 'datetime64[ns]'
print(test['ad_created_on'].dtype)   # Should print 'datetime64[ns]'

# Extract features: Use `.dt` accessor only if it's datetime
if pd.api.types.is_datetime64_any_dtype(train['ad_created_on']):
    max_date_train = train['ad_created_on'].max()
    train.loc[:, 'listing_age_days'] = (max_date_train - train['ad_created_on']).dt.days

    global_max = train['ad_created_on'].max()
    test['listing_age_days'] = (global_max - test['ad_created_on']).dt.days

else:
    print("Error: 'ad_created_on' is not in datetime format.")

In [ ]:
# Drop the 'ad_created_on' column afterward
train = train.drop(columns=["ad_created_on"])
test = test.drop(columns=["ad_created_on"])

# Leak features
leak_features = ["booking_down_pymnt", "emi_starts_from", "broker_quote"]
train = train.drop(columns=leak_features)
test = test.drop(columns=leak_features)

# Split data into features (X) and target (y)
X_train = train.drop(columns='sale_price')
y_train = train['sale_price']
X_test = test.drop(columns='sale_price')
y_test = test['sale_price']

# Define numeric and categorical features
num_features = X_train.select_dtypes(include='number').columns
df_obj = X_train.select_dtypes(include='object')

# Separate categorical features into those with <10 unique values (df_obj_one) and others (df_obj_target)
df_obj_one = []
for col in df_obj.columns:
    unique_vals = X_train[col].dropna().unique()
    n_unique = len(unique_vals)
    if n_unique < 10:
        df_obj_one.append(col)

# Categorical columns with >10 unique values
df_obj_target = [col for col in df_obj.columns if col not in df_obj_one]


In [ ]:
# Define preprocessing pipelines
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline_one = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # ('one', OneHotEncoder(handle_unknown='ignore', drop='first'))
    ('one', OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10))
])

cat_pipeline_target = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('target', TargetEncoder(handle_unknown='value', smoothing=5)),
    ('final_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Combine the pipelines in a ColumnTransformer
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, num_features),
    ('cat_one', cat_pipeline_one, df_obj_one),
    ('cat_target', cat_pipeline_target, df_obj_target)
])

# Define models
models = {
    "Random Forest": RandomForestRegressor(),
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(),
    "Gradient_Boost": GradientBoostingRegressor()
}

# Create pipelines for each model
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}


In [ ]:
X_train.columns

In [ ]:
X_test.columns

In [ ]:
X_test

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
X_test_transformed = preprocessor.transform(X_test)

print("Train NaNs:", pd.DataFrame(X_train_transformed).isna().sum().sum())
print("Test NaNs :", pd.DataFrame(X_test_transformed).isna().sum().sum())


# PHD LEVEL SHIT HERE:

Short answer: **Yes.**
Because you did a **time-based split**, `customer_to_customer` exists **only in the future (test period)** and **never appeared in training**, so the pipeline cannot learn a column for it. Therefore it cannot be transformed into a learned feature.

Now the precise mechanics.

---

# 1) What your pipeline actually learns

You fit the OneHotEncoder on **training data only**:

```python
pipe.fit(X_train, y_train)
```

Given your counts:

```
inperson_sale           6834
online                   397
customer_to_customer       43
```

But **time-based split means these counts are NOT from training**.
The true split distribution likely looks like:

### Training period

```
inperson_sale   ~6800
online          ~390
customer_to_customer   0   ❌
```

### Test period (future)

```
customer_to_customer   43   ✅ appears for first time
```

So during `.fit()`:

```python
ohe.categories_['source'] = ['inperson_sale', 'online']
```

`customer_to_customer` literally does not exist in the encoder vocabulary.

---

# 2) Why test data is NOT transformed into a new column

**OneHotEncoder is NOT a neural net. It does not learn patterns dynamically.**

It creates **fixed column schema at fit time**:

```
source_inperson_sale
source_online
```

When it sees:

```
source = customer_to_customer
```

It has only two legal columns, so it encodes:

```
0, 0
```

Because you told it:

```python
handle_unknown='ignore'
```

So:

> Pipeline transforms test data, but cannot invent new columns.

---

# 3) Why sklearn warns you

```
Found unknown categories
```

It is telling you:

> Your test distribution contains categories absent in training → distribution shift.

This is **not a sklearn bug**.
This is a **real-world ML problem (concept drift / covariate shift).**

---

# 4) Why this happens ONLY with time-based split

If you had random split:

```python
train_test_split(shuffle=True)
```

Then:

```
customer_to_customer appears in both train and test
```

So no warning.

But **time split preserves reality**, so it reveals **business evolution**.

This is why time-split is *harder but correct*.

---

# 5) Why this matters scientifically

Your model **never learned how customer_to_customer affects price**.

So predictions for these cars are **out-of-distribution**.

In production, this is dangerous.

---

# 6) Correct Industry Solutions

## Option A (Fast pragmatic fix)

Bucket rare categories BEFORE split:

```python
df['source'] = df['source'].replace('customer_to_customer', '__OTHER__')
```

---

## Option B (Best for structured data)

Use Target Encoding (time-aware):

```python
TargetEncoder(handle_unknown='value')
```

This **will give a numeric value even for unseen categories**.

---

## Option C (Production ML)

Retrain model periodically when new category share > threshold
(Drift monitoring).

---

# 7) Key ML Insight (Important)

Your dataset shows **business process change**:

```
customer_to_customer introduced later
```

This is **non-stationary data**.

Many ML models assume:

```
P_train(X) == P_test(X)
```

Your data violates this.

---

# 8) Should you worry?

For Kaggle / research → Yes, must handle.
For academic learning → This is an excellent real-world scenario.

---

# 9) Minimal fix for your current pipeline

```python
cat_pipeline_one = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=20,
        sparse_output=False
    ))
])
```

This auto-buckets `customer_to_customer`.

---

# 10) Critical concept (you just discovered)

You encountered:

**Temporal Covariate Shift**

This is PhD-level ML research topic.

---

# If you want

I can:

* simulate how much your R² drops when fixing leakage
* show how to do **time-aware target encoding properly**
* build a production-grade drift-safe pipeline
* explain how Netflix / Uber handle new categories

Just say the word.


You hit:

Concept drift

Distribution shift

Time-aware encoding

Leakage prevention

These are graduate-level ML engineering topics.

OneHotEncoder = dictionary frozen in time

TargetEncoder = statistical estimator that generalizes

Time-aware encoding = causality constraint

In [ ]:
# Make sure the preprocessor is fitted before accessing its transformers
if hasattr(preprocessor, 'transformers_'):
    # Find the columns passed to the 'cat_one' transformer
    for name, transformer, cols in preprocessor.transformers:
        if name == 'cat_one':
            cat_one_cols = cols
            break

    # Assuming the 'cat_one' transformer has a OneHotEncoder inside
    if 'cat_one' in preprocessor.named_transformers_:
        ohe = preprocessor.named_transformers_['cat_one'].named_steps['one']

        # Check which categories in X_test are unseen
        for i, col in enumerate(cat_one_cols):
            known = set(ohe.categories_[i])
            unseen = set(X_test[col].dropna().unique()) - known
            if unseen:
                print(f"OneHotEncoder input feature index {i} -> column '{col}' has unseen categories: {unseen}")


In [ ]:
# pipe = pipelines["Gradient_Boost"]  # any trained pipeline
# prep = pipe.named_steps["preprocessor"]

# for name, trans, cols in prep.transformers_:
#     print(name, cols)

# # Now inspect OHE
# cat_one_cols = prep.named_transformers_['cat_one'][2]
# ohe = prep.named_transformers_['cat_one'].named_steps['one']

# for i, col in enumerate(cat_one_cols):
#     known = set(ohe.categories_[i])
#     unseen = set(X_test[col].dropna().unique()) - known
#     if unseen:
#         print(f"{col} -> unseen categories: {unseen}")


In [ ]:
# Extract cat_one columns properly
for name, trans, cols in prep.transformers_:
    if name == "cat_one":
        cat_one_cols = cols
        break

# Get OHE inside pipeline
ohe = prep.named_transformers_['cat_one'].named_steps['one']

# Check unseen categories
for i, col in enumerate(cat_one_cols):
    known = set(ohe.categories_[i])
    unseen = set(X_test[col].dropna().unique()) - known
    if unseen:
        print(f"{col} -> unseen categories: {unseen}")


In [ ]:
pd.DataFrame(X_train_transformed)

In [ ]:
pd.DataFrame(X_test_transformed)

In [ ]:

# Function to evaluate the models
def get_regression_metrics(y_true, y_pred, model_name=None, verbose=True, plot=True):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    }

    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"MAE  : {mae:.2f}")
        print(f"MSE  : {mse:.2f}")
        print(f"RMSE : {rmse:.2f}")
        print(f"R2   : {r2:.4f}")
        print("-"*40)

    if plot:
        plt.figure(figsize=(16,5))

        # 1️⃣ True vs Predicted Scatter
        plt.subplot(1,2,1)
        sns.scatterplot(x=y_true, y=y_pred, alpha=0.6)
        plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', linewidth=2)
        plt.xlabel("Actual Values")
        plt.ylabel("Predicted Values")
        plt.title(f"Actual vs Predicted {'(' + model_name + ')' if model_name else ''}")

        # 2️⃣ Residual Plot
        plt.subplot(1,2,2)
        residuals = y_true - y_pred
        sns.histplot(residuals, kde=True, bins=30, color='orange')
        plt.xlabel("Residuals")
        plt.title(f"Residuals Distribution {'(' + model_name + ')' if model_name else ''}")

        plt.tight_layout()
        plt.show()

    return metrics

In [ ]:

# Store results in a dictionary
results = {}

# Train and evaluate each model
for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)

    # -------------------------
    # TRAIN metrics
    # -------------------------
    y_train_pred = pipe.predict(X_train)
    train_metrics = get_regression_metrics(
        y_train,
        y_train_pred,
        model_name=f"{name} (Train)",
        verbose=True,
        plot=True
    )

    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=True
    )

    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }

# Print results summary (optional)
for model_name, metrics in results.items():
    print(f"\n{model_name} Results:")
    print(f"Train Metrics: {metrics['train']}")
    print(f"Test Metrics: {metrics['test']}")

In [ ]:

# Store results in a dictionary
results = {}

# Train and evaluate each model
for name, pipe in pipelines.items():
    print(f"Training {name}...\n")

    # Train the model
    pipe.fit(X_train, y_train)


    # -------------------------
    # TEST metrics
    # -------------------------
    y_test_pred = pipe.predict(X_test)
    test_metrics = get_regression_metrics(
        y_test,
        y_test_pred,
        model_name=f"{name} (Test)",
        verbose=True,
        plot=False
    )

    # Store metrics in dictionary
    results[name] = {
        'train': train_metrics,
        'test': test_metrics
    }

# Print results summary (optional)
for model_name, metrics in results.items():
    print(f"\n{model_name} Results:")
    print(f"Train Metrics: {metrics['train']}")
    print(f"Test Metrics: {metrics['test']}")

In [ ]:
# Decision Tree Regressor
dt_params = {
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["squared_error", "friedman_mse", "absolute_error"]
}

# Random Forest Regressor
rf_params = {
    "n_estimators": [100, 200, 500, 1000],
    "max_depth": [None, 5, 8, 10, 15],
    "max_features": ["sqrt", "log2", 5, 7, 8],
    "min_samples_split": [2, 8, 15, 20],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}
gb_params = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None]
}

ada_params = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "estimator__max_depth": [1, 2, 3, 4, 5],
    "estimator__min_samples_leaf": [1, 2, 4, 8]
}


In [ ]:
ada_base = DecisionTreeRegressor(random_state=42)

randomcv_models = [
    ("Decision Tree", DecisionTreeRegressor(random_state=42), dt_params),
    ("Random Forest", RandomForestRegressor(random_state=42), rf_params),
    ("Ada_Boost",AdaBoostRegressor(estimator=ada_base, random_state=42),ada_params),
    ("Gradient_Boost",GradientBoostingRegressor(),gb_params)
]

# store fitted best estimators and best params
fitted_best_estimators = {}
best_params = {}

for name, model, params in randomcv_models:
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    params_prefixed = {f"model__{k}": v for k, v in params.items()}

    random_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=params_prefixed,
        n_iter=50,
        cv = TimeSeriesSplit(n_splits=3),
        verbose=2,
        n_jobs=-1,
        random_state=42,
        scoring='r2',
        refit=True  # ensure best_estimator_ is fitted on the whole X_train
    )

    random_search.fit(X_train, y_train)       # fitted on X_train folds
    print(f"BEST for {name}: {random_search.best_params_}")
    best_params[name] = random_search.best_params_
    # best_estimator_ is already a Pipeline fitted on the entire X_train
    fitted_best_estimators[name] = random_search.best_estimator_



In [ ]:
# Evaluate each fitted best estimator
results = {}
for name, fitted_pipe in fitted_best_estimators.items():
    print(f"\nEvaluating {name} ...")

    # Predictions on train and test
    y_train_pred = fitted_pipe.predict(X_train)
    y_test_pred  = fitted_pipe.predict(X_test)

    train_metrics = get_regression_metrics(y_train, y_train_pred, model_name=f"{name} (Train)", verbose=True, plot=False)
    test_metrics  = get_regression_metrics(y_test,  y_test_pred,  model_name=f"{name} (Test)",  verbose=True, plot=True)

    results[name] = {'best_params': best_params[name], 'train': train_metrics, 'test': test_metrics}


In [ ]:
# Evaluate each fitted best estimator
results = {}
for name, fitted_pipe in fitted_best_estimators.items():
    print(f"\nEvaluating {name} ...")

    # Predictions on train and test
    y_train_pred = fitted_pipe.predict(X_train)
    y_test_pred  = fitted_pipe.predict(X_test)

    train_metrics = get_regression_metrics(y_train, y_train_pred, model_name=f"{name} (Train)", verbose=True, plot=False)
    test_metrics  = get_regression_metrics(y_test,  y_test_pred,  model_name=f"{name} (Test)",  verbose=True, plot=False)

    results[name] = {'best_params': best_params[name], 'train': train_metrics, 'test': test_metrics}


In [ ]:
#Save the fitted best estimators
import joblib
for name, fitted in fitted_best_estimators.items():
    joblib.dump(fitted, f"model_{name.replace(' ', '_')}.pkl")


In [ ]:
# Parity & residual diagnostics (inspect failure modes)
import matplotlib.pyplot as plt
def parity_and_residuals(y_true, y_pred, title="Model"):
    resid = y_true - y_pred
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    plt.scatter(y_true, y_pred, alpha=0.4)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
    plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title(f"Parity ({title})")
    plt.subplot(1,2,2)
    plt.hist(resid, bins=50)
    plt.title("Residuals distribution"); plt.xlabel("residual")
    plt.tight_layout(); plt.show()

best = fitted_best_estimators["Gradient_Boost"]
y_test_pred = best.predict(X_test)
parity_and_residuals(y_test, y_test_pred, "Gradient_Boost (Test)")


Error-by-segment (find where model struggles)

Pick segment columns (e.g., make, model, city) and compute MAE/RMSE per group:

In [ ]:
X_test_copy = X_test.copy()
X_test_copy['y_true'] = y_test
X_test_copy['y_pred'] = y_test_pred

for seg in ['make','model','city']:
    if seg in X_test_copy.columns:
        grp = X_test_copy.groupby(seg).agg(
            n = ('y_true','size'),
            mae = ('y_true', lambda a: (a - X_test_copy.loc[a.index,'y_pred']).abs().mean())
        ).sort_values('n', ascending=False).head(10)
        print(seg); display(grp)


In [ ]:
def get_feature_names_from_column_transformer(ct, input_features):
    # adapted helper: returns final feature names after ColumnTransformer
    feature_names = []
    for name, trans, cols in ct.transformers_:
        if name == "remainder" and trans == "drop":
            continue
        if name == "remainder" and trans == "passthrough":
            feature_names.extend(cols)
            continue
        if hasattr(trans, "named_steps"):  # pipeline
            last = trans.named_steps[list(trans.named_steps.keys())[-1]]
            if hasattr(last, 'get_feature_names_out'):
                try:
                    names = last.get_feature_names_out(cols)
                except:
                    names = last.get_feature_names_out()
                feature_names.extend([f"{name}__{n}" for n in names])
            else:
                feature_names.extend([f"{name}__{c}" for c in cols])
        else:
            # transformer itself has no named_steps
            if hasattr(trans, 'get_feature_names_out'):
                feature_names.extend(list(trans.get_feature_names_out(cols)))
            else:
                feature_names.extend([f"{name}__{c}" for c in cols])
    return feature_names

# Example: for the best fitted pipeline
prep = fitted_best_estimators["Gradient_Boost"].named_steps['preprocessor']
feat_names = get_feature_names_from_column_transformer(prep, X_train.columns)
model = fitted_best_estimators["Gradient_Boost"].named_steps['model']

import pandas as pd
if hasattr(model, "feature_importances_"):
    fi = pd.Series(model.feature_importances_, index=feat_names).sort_values(ascending=False)
    display(fi.head(30))
else:
    print("Model has no feature_importances_. Consider SHAP.")


Optuna or BayesSearchCV for efficient hyperparameter optimization.

Cross-fold target encoding with time-aware folds (or prefer CatBoost).

SHAP to understand model decisions and detect bad features / leakage.

Drift monitoring if you plan to deploy (watch new categories like customer_to_customer).

Calibration of residuals (quantile regression) if you want prediction intervals.

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

estimators = [
    ('gb', fitted_best_estimators['Gradient_Boost'].named_steps['model']),
    ('rf', fitted_best_estimators['Random Forest'].named_steps['model'])
]

stack_pipe = Pipeline([
    ('preprocessor', preprocessor),   # same preprocessor
    ('stack', StackingRegressor(estimators=estimators, final_estimator=Ridge()))
])
stack_pipe.fit(X_train, y_train)
print("Stack test R2:", stack_pipe.score(X_test, y_test))


In [ ]:
%pip install -r C:\xtra\Last_Chance\git_re\inexorable-ML\requirements.txt